# BatchNorm và dropout bằng numpy, khớp PyTorch

Hai layer có hành vi **khác nhau giữa train và eval**, và vì vậy là nguồn bug kinh điển: quên `eval()`
khi đánh giá, running var sai một chút, mask dropout sinh lại trong backward. Bạn tự viết
`BatchNorm1d` và `Dropout` bằng numpy, đối chiếu từng con số với `torch.nn.BatchNorm1d`, rồi ghép vào MLP
trên MNIST để thấy chúng làm gì với đường cong train.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
  Cell đầu tiên tải MNIST vào `data/raw/mnist` nếu chưa có.
- Input của cả hai layer có shape `(N, D)`: batch theo dòng, như phần lý thuyết.
- Chấm: `uv run pytest tracks/ai-engineer/modules/01-nn-from-scratch/exercises/intermediate/03-batchnorm-dropout -q`

In [ ]:
# explore: giới hạn luồng BLAS, tìm gốc repo, tải MNIST nếu chưa có (khoảng 11 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy; ma trận nhỏ mà 16 luồng thì chậm hơn ~5 lần
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

MNIST = ensure("mnist")
print(MNIST, sorted(p.name for p in MNIST.iterdir()))

In [ ]:
import gzip
import time

import numpy as np

## 1. BatchNorm1d

`BatchNorm1d(num_features, momentum=0.1, eps=1e-5)`, công thức ở mục 7 lý thuyết:

- Thuộc tính: `gamma` (khởi tạo 1), `beta` (0), `running_mean` (0), `running_var` (1), đều shape `(D,)`;
  `dgamma`, `dbeta` được **gán** mỗi lần `backward`.
- `forward(x, training=True)`: chuẩn hoá bằng mean và phương sai **lệch** (chia $$N$$) của batch, rồi
  $$y = \gamma\hat x + \beta$$. Cập nhật $$\mu_{run} \leftarrow (1-m)\mu_{run} + m\mu$$ và tương tự cho running var, nhưng
  với phương sai **không lệch** (chia $$N-1$$), đúng như PyTorch. Batch dưới 2 mẫu thì `raise ValueError`.
- `forward(x, training=False)`: dùng running stats, không đổi chúng.
- `backward(grad)`: theo chế độ của lần `forward` gần nhất. Khi train, $$\mu$$ và $$\sigma^2$$ phụ thuộc mọi mẫu
  trong batch, nên gradient theo $$x$$ có thêm hai số hạng; khi eval, chúng là hằng số.

In [ ]:
class BatchNorm1d:
    """BatchNorm cho input (N, D), khớp torch.nn.BatchNorm1d (affine, track_running_stats).

    gamma (ones), beta (zeros): tham số học. running_mean (zeros), running_var (ones): thống kê cho eval.
    dgamma, dbeta: gradient, được GÁN mỗi lần gọi backward.
    """

    def __init__(self, num_features: int, momentum: float = 0.1, eps: float = 1e-5):
        # TODO
        raise NotImplementedError

    def forward(self, x: np.ndarray, training: bool) -> np.ndarray:
        """training: chuẩn hoá bằng thống kê của batch và cập nhật running stats. eval: dùng running stats."""
        # TODO
        raise NotImplementedError

    def backward(self, grad: np.ndarray) -> np.ndarray:
        """grad = dL/d(out) shape (N, D). Gán self.dgamma, self.dbeta; trả dL/dx theo chế độ của lần forward trước."""
        # TODO
        raise NotImplementedError

## 2. Dropout

`Dropout(p, rng)`, **inverted dropout**:

- `forward(x, training=True)`: mỗi phần tử bị đặt 0 với xác suất $$p$$, phần còn lại nhân $$1/(1-p)$$ để kỳ vọng
  không đổi. Mask sinh **mới mỗi lần forward** từ `rng` và được lưu lại.
- `forward(x, training=False)` (và `p = 0`): trả nguyên `x`.
- `backward(grad)`: dùng đúng mask của lần forward trước (eval thì trả nguyên `grad`).
- `p` ngoài khoảng $$[0, 1)$$ thì `raise ValueError`.

In [ ]:
class Dropout:
    """Inverted dropout với xác suất bỏ p; mọi số ngẫu nhiên lấy từ rng."""

    def __init__(self, p: float, rng: np.random.Generator):
        # TODO
        raise NotImplementedError

    def forward(self, x: np.ndarray, training: bool) -> np.ndarray:
        # TODO
        raise NotImplementedError

    def backward(self, grad: np.ndarray) -> np.ndarray:
        # TODO
        raise NotImplementedError

In [ ]:
# explore: đối chiếu với torch.nn.BatchNorm1d trên một batch lệch mean và scale
import torch

rng = np.random.default_rng(0)
x = 3.0 + 2.0 * rng.normal(size=(16, 4))
bn, ref = BatchNorm1d(4), torch.nn.BatchNorm1d(4).double()
out = bn.forward(x, training=True)
ref_out = ref(torch.tensor(x)).detach().numpy()
print("train: max |out - torch|", np.abs(out - ref_out).max())
print("running_var:", bn.running_var.round(4), "| torch:", ref.running_var.numpy().round(4))
ref.eval()
x2 = rng.normal(size=(3, 4))
print("eval: max |out - torch|", np.abs(bn.forward(x2, training=False) - ref(torch.tensor(x2)).detach().numpy()).max())

drop = Dropout(0.3, np.random.default_rng(0))
y = drop.forward(np.ones((1000, 1000)), training=True)
print(f"dropout p=0.3: mean {y.mean():.4f}, tỉ lệ bằng 0 {np.mean(y == 0):.4f}")

## 3. Ghép vào MLP trên MNIST

Các cell dưới dùng `Linear`, `ReLU`, cross-entropy theo lời giải mẫu Basic 03 (float32; thay bằng code
của bạn nếu muốn) và hai lớp của bạn để train MLP 784-256-256-10 trên **10 000** ảnh train (ít dữ liệu để thấy overfit), 15 epoch, SGD lr
0.1, batch 64, cùng seed cho bốn cấu hình. Cả phần này chạy khoảng 15 giây.

In [ ]:
# explore: Linear, ReLU, cross-entropy theo lời giải mẫu Basic 03 (float32), thay bằng code của bạn nếu muốn; hàm tiện ích để ghép MLP
class Linear:
    def __init__(self, in_features, out_features, rng):
        self.W = rng.normal(0, np.sqrt(2 / in_features), (in_features, out_features)).astype(np.float32)
        self.b = np.zeros(out_features, np.float32)
        self.dW, self.db = np.zeros_like(self.W), np.zeros_like(self.b)

    def forward(self, x, training=True):
        self.x = x
        return x @ self.W + self.b

    def backward(self, grad):
        self.dW, self.db = self.x.T @ grad, grad.sum(axis=0)
        return grad @ self.W.T


class ReLU:
    def forward(self, x, training=True):
        self.mask = x > 0
        return x * self.mask

    def backward(self, grad):
        return grad * self.mask


def cross_entropy(logits, y):
    """(loss trung bình, dL/dlogits)."""
    z = logits - logits.max(axis=1, keepdims=True)
    log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    rows = np.arange(len(y))
    g = np.exp(log_p)
    g[rows, y] -= 1
    return float(-log_p[rows, y].mean()), g / len(y)


def build(bn: bool, p: float, hidden: int = 256, seed: int = 0) -> list:
    """784 → [Linear → (BN) → ReLU → (Dropout)] × 2 → Linear(10)."""
    rng, layers, n_in = np.random.default_rng(seed), [], 784
    for _ in range(2):
        layers.append(Linear(n_in, hidden, rng))
        if bn:
            layers.append(BatchNorm1d(hidden))
        layers.append(ReLU())
        if p:
            layers.append(Dropout(p, rng))
        n_in = hidden
    layers.append(Linear(n_in, 10, rng))
    return layers


def forward(layers, x, training):
    for layer in layers:
        x = layer.forward(x, training)
    return x


def sgd(layers, lr):
    for layer in layers:
        if isinstance(layer, Linear):
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db
        elif isinstance(layer, BatchNorm1d):
            layer.gamma -= lr * layer.dgamma
            layer.beta -= lr * layer.dbeta


def accuracy(layers, X, y, training=False, batch_size=1000):
    hits = [forward(layers, X[i:i + batch_size], training).argmax(axis=1) == y[i:i + batch_size]
            for i in range(0, len(X), batch_size)]
    return float(np.concatenate(hits).mean())

In [ ]:
# explore: đọc MNIST (idx gz). Train trên 10 000 ảnh đầu để overfit hiện rõ, đánh giá trên 10 000 ảnh test
def load_idx(path):
    with gzip.open(path) as f:
        data = f.read()
    ndim = data[3]
    shape = np.frombuffer(data, ">u4", ndim, offset=4)
    return np.frombuffer(data, np.uint8, offset=4 + 4 * ndim).reshape(shape)

X_all = load_idx(MNIST / "train-images-idx3-ubyte.gz").reshape(-1, 784).astype(np.float32) / 255
y_all = load_idx(MNIST / "train-labels-idx1-ubyte.gz").astype(np.int64)
X_test = load_idx(MNIST / "t10k-images-idx3-ubyte.gz").reshape(-1, 784).astype(np.float32) / 255
y_test = load_idx(MNIST / "t10k-labels-idx1-ubyte.gz").astype(np.int64)
X_train, y_train = X_all[:10000], y_all[:10000]
print(X_train.shape, X_test.shape)

In [ ]:
# explore: 4 cấu hình × 15 epoch, SGD lr 0.1, batch 64, cùng seed: có/không BatchNorm, có/không dropout 0.3
EPOCHS, LR = 15, 0.1
results = {}
for bn, p in [(False, 0.0), (True, 0.0), (False, 0.3), (True, 0.3)]:
    name = f"{'BN' if bn else 'không BN'}, dropout {p}"
    layers, rng = build(bn, p), np.random.default_rng(0)
    losses, test_accs, start = [], [], time.perf_counter()
    for epoch in range(EPOCHS):
        idx, total = rng.permutation(len(X_train)), 0.0
        for i in range(0, len(X_train), 64):
            b = idx[i:i + 64]
            loss, g = cross_entropy(forward(layers, X_train[b], True), y_train[b])
            for layer in reversed(layers):
                g = layer.backward(g)
            sgd(layers, LR)
            total += loss * len(b)
        losses.append(total / len(X_train))
        test_accs.append(accuracy(layers, X_test, y_test))   # eval mode
    results[name] = (layers, losses, test_accs)
    print(f"{name:22s} train loss cuối {losses[-1]:.3f} | train acc {accuracy(layers, X_train, y_train):.4f}"
          f" | test acc epoch 1 {test_accs[0]:.4f}, cuối {test_accs[-1]:.4f} | {time.perf_counter() - start:.1f} s")

In [ ]:
# explore: đường cong train loss (trái, thang log) và test accuracy (phải) theo epoch
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, (_, losses, test_accs) in results.items():
    ax1.plot(range(1, EPOCHS + 1), losses, marker=".", label=name)
    ax2.plot(range(1, EPOCHS + 1), test_accs, marker=".", label=name)
ax1.set_yscale("log"); ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss")
ax2.set_xlabel("epoch"); ax2.set_ylabel("test accuracy"); ax2.legend()
plt.show()

In [ ]:
# explore: quên chuyển sang eval. Đánh giá model "BN, dropout 0.3" ở chế độ train, rồi xem running stats.
layers = results["BN, dropout 0.3"][0]
print(f"eval mode                     : {accuracy(layers, X_test, y_test):.4f}")
print(f"train mode, batch 1000        : {accuracy(layers, X_test, y_test, training=True):.4f}")
print(f"train mode, batch 4           : {accuracy(layers, X_test, y_test, training=True, batch_size=4):.4f}")
# Mỗi forward ở chế độ train cũng cập nhật running stats: 2 500 batch 4 ảnh test đã ghi đè thống kê train
print(f"eval mode lại, sau hai lượt trên: {accuracy(layers, X_test, y_test):.4f}")

## 4. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. BatchNorm đổi gì ở **epoch đầu**, và đổi gì ở **cuối** 15 epoch? Hai điều đó có mâu thuẫn không?
2. Dropout đổi khoảng cách giữa train và test thế nào? Nó có làm test accuracy tốt hơn hẳn ở đây không,
   và vì sao điều đó không có nghĩa dropout vô dụng?
3. Ở cell cuối, vì sao đánh giá ở chế độ train với batch 4 tệ hơn hẳn batch 1000? Vì sao sau đó chuyển
   lại sang eval mà accuracy vẫn không như cũ?